In [62]:
"""
Makes a catalogue of morphological data for my galaxies. Creates fits data file Galaxy_morphology.
"""

from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import os
import corner
import asdf
from tqdm import tqdm
from photutils.aperture import EllipticalAperture

with fits.open("/nvme/scratch/work/alberttg/Summer_project/Ha_and_NII_broad_line_data.fits") as hdul:
        data = hdul[1].data
TABLE = Table(data)


GALAXY_ID = TABLE["SURVEY_ID"]        # object ID, used for labeling/output files
SERSIC_FILTERS = ["F444W","F356W","F277W"]                    # filter name, used for labeling/output files
SURVEY = TABLE["SURVEY"]

CUTOUT_SIZE = 0.96 #as
PIXEL_SIZE = 0.03 #as



In [63]:
from astropy.visualization import ImageNormalize, AsinhStretch, PercentileInterval

def plot_science_model_residual(
    science,
    model,
    residual=None,
    percentile=99.5,
    cmap="gray",
    residual_cmap="RdBu_r",
    figsize=(12, 4),
):
    """
    Plot science image, best-fit model, and residual side-by-side.
    I only needed this because the double sersic run didn't save any images.

    Parameters
    ----------
    science : 2D ndarray
        Science image.
    model : 2D ndarray
        Best-fit model image.
    residual : 2D ndarray, optional
        Residual image. If None, computed as science - model.
    percentile : float
        Percentile used for image normalization.
    cmap : str
        Colormap for science/model.
    residual_cmap : str
        Colormap for residual.
    figsize : tuple
        Figure size.
    """

    if residual is None:
        residual = science - model

    # Shared normalization for science/model
    interval = PercentileInterval(percentile)
    vmin, vmax = interval.get_limits(
        np.concatenate([science.ravel(), model.ravel()])
    )

    norm = ImageNormalize(
        vmin=vmin,
        vmax=vmax,
        stretch=AsinhStretch(),
    )

    # Symmetric residual scaling
    rlim = np.nanpercentile(np.abs(residual), percentile)

    fig, axes = plt.subplots(
        1, 3,
        figsize=figsize,
        constrained_layout=True,
    )

    images = [
        (science, "Science", cmap, norm),
        (model, "Best Model", cmap, norm),
        (
            residual,
            "Residual",
            residual_cmap,
            dict(vmin=-rlim, vmax=rlim),
        ),
    ]

    for ax, (img, title, cmap_i, norm_i) in zip(axes, images):

        if isinstance(norm_i, dict):
            im = ax.imshow(
                img,
                origin="lower",
                cmap=cmap_i,
                **norm_i,
            )
        else:
            im = ax.imshow(
                img,
                origin="lower",
                cmap=cmap_i,
                norm=norm_i,
            )

        ax.set_title(title, fontsize=12)
        ax.set_xticks([])
        ax.set_yticks([])

        plt.colorbar(
            im,
            ax=ax,
            fraction=0.046,
            pad=0.04,
        )

    return fig, axes

In [64]:
def read_summary_table(path, parameter):
    """
    Extracts the mean and sd values for a given parameter from the sersic summary tables.
    """
    csv_table = pd.read_csv(path, index_col=0)
    
    row = csv_table.loc[parameter]
    return row["mean"], row["sd"]

In [65]:
def double_sersic_data(galaxy_id, filt, asdf_path, output_dir):
    """
    The double sersic fit run didn't save the data/model/residual fits file or any images. Don't need this anymore.
    """

    os.makedirs(output_dir, exist_ok=True)
    tag = f"{galaxy_id}_{filt}"

    with asdf.open(asdf_path) as af:
        model = np.asarray(af.tree['best_model'])
        image = np.asarray(af.tree["input_data"]['image'])
        mask = np.asarray(af.tree["input_data"]['mask'])
        rms = np.asarray(af.tree["input_data"]['rms'])

        fits_path = os.path.join(output_dir, f"{tag}_data_model_residual.fits")

        residual = image - model

        hdul = fits.HDUList([
            fits.PrimaryHDU(),                              # Empty primary HDU
            fits.ImageHDU(image, name="SCIENCE_IMAGE"),         # Extension 1
            fits.ImageHDU(model, name="MODEL"),             # Extension 2
            fits.ImageHDU(residual, name="RESIDUAL"),       # Extension 3
            fits.ImageHDU(mask.astype("uint8"), name="MASK"),  # Extension 4
            fits.ImageHDU(rms, name="RMS"),                 # Extension 5
        ])
        # Fits path
        hdul.writeto(fits_path, overwrite=True)
        hdul.close()
        print(f"[{tag}] Saved FITS residual file to {fits_path}")

        # residual figure
        fig_resid, _ = plot_science_model_residual(image, model, residual)
        fig_resid.suptitle(f"{galaxy_id} - {filt}: data / model / residual")
        resid_path = os.path.join(output_dir, f"{tag}_data_model_residual.png")
        fig_resid.savefig(resid_path, dpi=150, bbox_inches="tight")
        print(f"[{tag}] Saved data/model/residual plot to {resid_path}")
        plt.close(fig_resid)

        # Corner plot
        posterior = af.tree["posterior"]

        labels = list(posterior.keys())

        samples = np.column_stack([
            np.asarray(posterior[p]).reshape(-1)
            for p in labels
        ])

        fig_corner = corner.corner(
            samples,
            labels=labels,
            show_titles=True,
            title_fmt=".3f",
        )

        fig_corner.suptitle(f"{galaxy_id} - {filt}: posterior corner plot")
        corner_path = os.path.join(output_dir, f"{tag}_corner.png")
        fig_corner.savefig(corner_path, dpi=150, bbox_inches="tight")
        plt.close(fig_corner)


In [66]:
def read_residual_fits_table(path):
    """
    Opens data_model_residual fits files and returns data for sersic model and residual.
    """
    with fits.open(path) as hdul:
        sci_im = hdul[1].data
        sersic_model = hdul[2].data
        residual = hdul[3].data
        mask = hdul[4].data
        mask = mask.astype(bool)
        rms = hdul[5].data

    return sci_im, sersic_model, residual, mask, rms

In [67]:
def calculate_RFF(sci_im, residual, mask, rms, flux_auto, flux_radius):
    """
    Formula for RFF from EPOCHS XI eq 4.

    Parameters
    ----------
    sci_im : array
        science image
    residual : array
        residual data (between sersic model and science image)
    mask : array
        mask map
    rms : array
        map of background noise
    flux_auto : float
        Flux of galaxy measured through SExtractor
    flux_radius : float
        Half-light radius measurement measured with SExtractor
    """

    ny, nx = sci_im.shape
    y0, x0 = ny // 2, nx // 2

    y, x = np.indices(sci_im.shape)

    r = np.sqrt((x - x0)**2 + (y - y0)**2)

    rff_region = r <= 2 * flux_radius
    # Defines what region is the galaxy and therefore where RFF can be meaningfully calculated

    # Remove contaminating sources
    good_pixels = rff_region & (~mask)

    # Number of pixels in aperture (RFF is calculated within twice flux_radius)
    N_pixels = np.sum(good_pixels)

    background_values = sci_im[~mask.astype(bool)]
    depth_1sig = 1.4826 * np.nanmedian(np.abs(background_values - np.nanmedian(background_values)))

    rff = ( np.sum(np.abs(residual[good_pixels])) - 0.8 * depth_1sig * N_pixels)  / flux_auto
    # Must restrict the residual map to the region where RFF is defined (twice flux_radius)

    return rff

In [68]:
def calculate_RFF_trial(sci_im, residual, mask, flux_auto, flux_radius, x0, y0):
    """
    Formula for RFF from EPOCHS XI eq 4.

    Parameters
    ----------
    sci_im : array
        science image
    residual : array
        residual data (between sersic model and science image)
    mask : array
        mask map
    rms : array
        map of background noise
    flux_auto : float
        Flux of galaxy measured through SExtractor
    flux_radius : float
        Half-light radius measurement measured with SExtractor
    """

    y, x = np.indices(sci_im.shape)

    r = np.sqrt((x - x0)**2 + (y - y0)**2)
    # Uses x0 and y0 from model

    rff_region = r <= 2 * flux_radius
    # Defines what region is the galaxy and therefore where RFF can be meaningfully calculated

    # Remove contaminating sources
    good_pixels = rff_region & (~mask)

    # Number of pixels in aperture (RFF is calculated within twice flux_radius)
    N_pixels = np.sum(good_pixels)

    background_values = sci_im[~mask.astype(bool)]
    depth_1sig = 1.4826 * np.nanmedian(np.abs(background_values - np.nanmedian(background_values)))

    rff = ( np.sum(np.abs(residual[good_pixels])) - 0.8 * depth_1sig * N_pixels)  / flux_auto
    # Must restrict the residual map to the region where RFF is defined (twice flux_radius)

    return rff

In [69]:
def _calc_RFF(sci_im, model, residuals, mask, a_image_as, b_image_as, theta_image):
        """
        Calculates the Residual Flux Fraction (RFF) for a given galaxy.

        Parameters
        ----------
        model : array
            Sersic model data
        residuals : array
            Residuals bewteen science image and model
        mask : array of bool
            Mask map
        a_image_as : float
            Semi major axis of elliptical aperture used in image (units of as)
        b_image_as : float
            Semi minor axis of elliptical aperture used in image (units of as)
        theta_image : float
            Angle of orientation of elliptical aperture used in image (degrees?)
        """
        # -> Kron radius too small
        # -> elliptical aperture also too small

        pix_scale = CUTOUT_SIZE / PIXEL_SIZE
        # reconstruct Kron elliptical aperture
        # EllipticalAperture models an elliptical aperture taking 5 parameters: x0, y0, a, b, orientation
        kron_aper = EllipticalAperture(
            (PIXEL_SIZE / 2, PIXEL_SIZE / 2),
            a_image_as / pix_scale,
            b_image_as / pix_scale,
            theta_image,
        )
        residuals = np.abs(residuals) 
        model_kron = kron_aper.do_photometry(model)[0][0]
        residual_kron = kron_aper.do_photometry(residuals)[0][0]

        # Mask all sources except those within Kron aperture
        mask[mask != 0] = 1
        background_values = sci_im[~mask.astype(bool)]
        abs_deviation = np.abs(background_values - np.nanmedian(background_values))
        depth_1sig = 1.4826 * np.nanmedian(abs_deviation)

        rff = (residual_kron - 0.8 * depth_1sig * kron_aper.area) / model_kron
        return rff

In [70]:
def calculate_BIC(sci_im, residual, mask, rms, flux_radius, n_params):
    """
    Calculates BIC, assuming Gaussian pixel uncertainties.
    """
    ny, nx = sci_im.shape
    y0, x0 = ny // 2, nx // 2

    y, x = np.indices(sci_im.shape)

    r = np.sqrt((x - x0)**2 + (y - y0)**2)

    region = r <= 2 * flux_radius
    # Defines what region is the galaxy and therefore where RFF can be meaningfully calculated
    # TODO: I'm not sure if this is appropriate for the BIC

    # Remove contaminating sources and ensure that rms is not Nan nor zero
    good_pixels = region & (~mask) & np.isfinite(rms) & (rms != 0)

    # Number of pixels in aperture (RFF is calculated within twice flux_radius)
    N_pixels = np.nansum(good_pixels)

    err = rms[good_pixels]
    # log-likelihood for Gaussian pixel uncertainties
    # TODO: not sure if thats appropriate here too
    ln_L = -0.5 * np.nansum( residual[good_pixels]**2 / err**2 + np.log(2 * np.pi * err**2) )

    bic = n_params * np.log(N_pixels) - 2 * ln_L

    return bic
    

In [71]:
def make_single_sersic_table(folder):
    """
    Makes a FITS table of morphological parameters for each galaxy.
    Each galaxy occupies a single row, with separate columns for each filter.
    """

    rows = []

    for i in tqdm(range(len(GALAXY_ID)), total=len(GALAXY_ID)):

        # Start the row with the galaxy information
        row = {
            "SURVEY_ID": GALAXY_ID[i],
            "SURVEY": SURVEY[i],
            "REDSHIFT": TABLE["REDSHIFT"][i],
        }

        # Add morphology measurements for each filter
        for filt in SERSIC_FILTERS:

            summary_path = Path(
                f"/nvme/scratch/work/alberttg/Summer_project/{folder}/"
                f"{GALAXY_ID[i]}/{GALAXY_ID[i]}_{filt}_summary.csv"
            )

            if summary_path.exists():
                ellip_mean, ellip_sd = read_summary_table(summary_path, "ellip")
                n_mean, n_sd = read_summary_table(summary_path, "n")
                r_eff_mean, r_eff_sd = read_summary_table(summary_path, "r_eff")
                xc, _ = read_summary_table(summary_path, "xc")
                yc, _ = read_summary_table(summary_path, "yc")
            else:
                ellip_mean = ellip_sd = np.nan
                n_mean = n_sd = np.nan
                r_eff_mean = r_eff_sd = np.nan
                xc = np.nan
                yc = np.nan

            row[f"{filt}_ellip_mean"] = ellip_mean
            row[f"{filt}_ellip_sd"] = ellip_sd
            row[f"{filt}_n_mean"] = n_mean
            row[f"{filt}_n_sd"] = n_sd
            row[f"{filt}_r_eff_mean"] = r_eff_mean
            row[f"{filt}_r_eff_sd"] = r_eff_sd

            fits_path = Path(f"/nvme/scratch/work/alberttg/Summer_project/{folder}/{GALAXY_ID[i]}/{GALAXY_ID[i]}_{filt}_data_model_residual.fits")

            if fits_path.exists():
                sci_im, sersic_model, residual, mask, rms \
                    = read_residual_fits_table(fits_path)
                
                flux_auto = f"FLUX_AUTO_{filt}"
                flux_radius = f"FLUX_RADIUS_{filt}"
                a_image = f"A_IMAGE_{filt}"
                b_image = f"B_IMAGE_{filt}"
                theta_image = f"THETA_IMAGE_{filt}"
                
                my_rff = calculate_RFF(sci_im, residual, mask, rms, TABLE[flux_auto][i], TABLE[flux_radius][i])

                rff = calculate_RFF_trial(sci_im, residual, mask, TABLE[flux_auto][i], TABLE[flux_radius][i], xc, yc)

                bic = calculate_BIC(sci_im, residual, mask, rms, TABLE[flux_radius][i], 8)

            else:
                rff = np.nan
                my_rff = np.nan

            row[f"{filt}_my_RFF"] = my_rff
            row[f"{filt}_RFF"] = rff
            row[f"{filt}_BIC"] = bic
        # Append one completed row per galaxy
        rows.append(row)

    new_table = Table(rows=rows)

    print(new_table.colnames)

    new_table.write(f"All_galaxies_{folder}_table.fits", format="fits", overwrite=True)

    return new_table

In [72]:
def make_double_sersic_table(folder):
    """
    Makes a FITS table of morphological parameters for each galaxy.
    Each galaxy occupies a single row, with separate columns for each filter.
    """

    rows = []

    for i in tqdm(range(len(GALAXY_ID)), total=len(GALAXY_ID)):

        # Start the row with the galaxy information
        row = {
            "SURVEY_ID": GALAXY_ID[i],
            "SURVEY": SURVEY[i],
            "REDSHIFT": TABLE["REDSHIFT"][i],
        }

        # Add morphology measurements for each filter
        for filt in SERSIC_FILTERS:

            summary_path = Path(
                f"/nvme/scratch/work/alberttg/Summer_project/{folder}/"
                f"{GALAXY_ID[i]}/{GALAXY_ID[i]}_{filt}_summary.csv"
            )

            if summary_path.exists():
                ellip_1_mean, ellip_1_sd = read_summary_table(summary_path, "ellip_1")
                n_1_mean, n_1_sd = read_summary_table(summary_path, "n_1")
                r_eff_1_mean, r_eff_1_sd = read_summary_table(summary_path, "r_eff_1")

                ellip_2_mean, ellip_2_sd = read_summary_table(summary_path, "ellip_2")
                n_2_mean, n_2_sd = read_summary_table(summary_path, "n_2")
                r_eff_2_mean, r_eff_2_sd = read_summary_table(summary_path, "r_eff_2")

                xc, _ = read_summary_table(summary_path, "xc")
                yc, _ = read_summary_table(summary_path, "yc")
            else:
                ellip_1_mean = ellip_1_sd = np.nan
                n_1_mean = n_1_sd = np.nan
                r_eff_1_mean = r_eff_1_sd = np.nan
                ellip_2_mean = ellip_2_sd = np.nan
                n_2_mean = n_2_sd = np.nan
                r_eff_2_mean = r_eff_2_sd = np.nan
                xc = np.nan
                yc = np.nan


            row[f"{filt}_ellip_1_mean"] = ellip_1_mean
            row[f"{filt}_ellip_1_sd"] = ellip_1_sd
            row[f"{filt}_n_1_mean"] = n_1_mean
            row[f"{filt}_n_1_sd"] = n_1_sd
            row[f"{filt}_r_eff_1_mean"] = r_eff_1_mean
            row[f"{filt}_r_eff_1_sd"] = r_eff_1_sd

            row[f"{filt}_ellip_2_mean"] = ellip_2_mean
            row[f"{filt}_ellip_2_sd"] = ellip_2_sd
            row[f"{filt}_n_2_mean"] = n_2_mean
            row[f"{filt}_n_2_sd"] = n_2_sd
            row[f"{filt}_r_eff_2_mean"] = r_eff_2_mean
            row[f"{filt}_r_eff_2_sd"] = r_eff_2_sd

            fits_path = Path(f"/nvme/scratch/work/alberttg/Summer_project/{folder}/{GALAXY_ID[i]}/{GALAXY_ID[i]}_{filt}_data_model_residual.fits")

            if fits_path.exists():
                sci_im, sersic_model, residual, mask, rms \
                    = read_residual_fits_table(fits_path)
                
                flux_auto = f"FLUX_AUTO_{filt}"
                flux_radius = f"FLUX_RADIUS_{filt}"
                a_image = f"A_IMAGE_{filt}"
                b_image = f"B_IMAGE_{filt}"
                theta_image = f"THETA_IMAGE_{filt}"
                
                my_rff = calculate_RFF(sci_im, residual, mask, rms, TABLE[flux_auto][i], TABLE[flux_radius][i])

                rff = calculate_RFF_trial(sci_im, residual, mask, TABLE[flux_auto][i], TABLE[flux_radius][i], xc, yc)
                
                bic = calculate_BIC(sci_im, residual, mask, rms, TABLE[flux_radius][i], 12)

            else:
                rff = np.nan
                my_rff = np.nan
                bic = np.nan

            row[f"{filt}_my_RFF"] = my_rff
            row[f"{filt}_RFF"] = rff
            row[f"{filt}_BIC"] = bic
        # Append one completed row per galaxy
        rows.append(row)

    new_table = Table(rows=rows)

    print(new_table.colnames)

    new_table.write(f"All_galaxies_{folder}_table.fits", format="fits", overwrite=True)

    return new_table

In [73]:
if __name__ == "__main__":
    single_table = make_single_sersic_table("Single_sersic_fits")
    # Posterior data on single sersic profile fitting
    # double_table = make_double_sersic_table("Double_sersic_fits")
    # Posterior data on double sersic profile fitting
    
    # single_PS_table
    

  1%|▏         | 2/144 [00:00<00:08, 17.45it/s]

100%|██████████| 144/144 [00:03<00:00, 41.33it/s]

['SURVEY_ID', 'SURVEY', 'REDSHIFT', 'F444W_ellip_mean', 'F444W_ellip_sd', 'F444W_n_mean', 'F444W_n_sd', 'F444W_r_eff_mean', 'F444W_r_eff_sd', 'F444W_my_RFF', 'F444W_RFF', 'F444W_BIC', 'F356W_ellip_mean', 'F356W_ellip_sd', 'F356W_n_mean', 'F356W_n_sd', 'F356W_r_eff_mean', 'F356W_r_eff_sd', 'F356W_my_RFF', 'F356W_RFF', 'F356W_BIC', 'F277W_ellip_mean', 'F277W_ellip_sd', 'F277W_n_mean', 'F277W_n_sd', 'F277W_r_eff_mean', 'F277W_r_eff_sd', 'F277W_my_RFF', 'F277W_RFF', 'F277W_BIC']


In [74]:
"""
for i in tqdm(range(len(GALAXY_ID)), total=len(GALAXY_ID)):
    for filt in SERSIC_FILTERS:
        asdf_path = Path(f'/nvme/scratch/work/alberttg/Summer_project/Double_sersic_fits/{GALAXY_ID[i]}/{filt}_sersic_fit_data.asdf')

        output_dir = f"/nvme/scratch/work/alberttg/Summer_project/Double_sersic_fits/{GALAXY_ID[i]}"

        if asdf_path.exists():
            double_sersic_data(GALAXY_ID[i], filt, asdf_path, output_dir)
        else:
            print(f"{asdf_path} does not exist")
"""
with fits.open('/nvme/scratch/work/alberttg/Summer_project/Double_sersic_fits/130917/130917_F277W_data_model_residual.fits') as hdul:
    hdul.info()
        

Filename: /nvme/scratch/work/alberttg/Summer_project/Double_sersic_fits/130917/130917_F277W_data_model_residual.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1  SCIENCE_IMAGE    1 ImageHDU         8   (32, 32)   float64   
  2  MODEL         1 ImageHDU         8   (32, 32)   float64   
  3  RESIDUAL      1 ImageHDU         8   (32, 32)   float64   
  4  MASK          1 ImageHDU         8   (32, 32)   uint8   
  5  RMS           1 ImageHDU         8   (32, 32)   float64   
